In [ ]:
%cd ../../..
import os
import torch
import polars as pl
import pydicom
import numpy as np
from tqdm import tqdm
from omegaconf import OmegaConf

from dinov2.inference import generate_embeddings, build_model, view_volume, load_dicom

In [ ]:
def get_metadata(data_path):
    metadata_raw = pl.read_csv(os.path.join(data_path, "NSCLC-Radiomics/metadata.csv"))
    metadata_raw = metadata_raw.select(["Subject ID", "Modality", "File Location"])

    ct_meta = metadata_raw.filter(pl.col("Modality") == "CT").drop("Modality")
    seg_meta = metadata_raw.filter(pl.col("Modality") == "SEG").drop("Modality")

    merged = ct_meta.join(seg_meta, on="Subject ID", how="inner", suffix="_SEG")

    merged = merged.rename(
        {"File Location": "CT File Location", "File Location_SEG": "SEG File Location"}
    )
    return merged

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets"

metadata = get_metadata(data_path)
metadata.head()

In [ ]:
def get_bbox(folder_path: str, label: str, pad=0):
    files = [
        os.path.join(folder_path, x)
        for x in os.listdir(folder_path)
        if x.endswith(".dcm")
    ]
    if len(files) != 1:
        raise RuntimeError(f"Expected to find exactly one file. Found: {len(files)}")

    ds = pydicom.dcmread(files[0])

    segments = ds.SegmentSequence
    per_frame = ds.PerFrameFunctionalGroupsSequence

    n_frames = ds.NumberOfFrames
    rows = ds.Rows
    cols = ds.Columns

    pixel_array = np.frombuffer(ds.PixelData, dtype=np.uint8)
    pixel_array = np.unpackbits(pixel_array)
    pixel_array = pixel_array[: n_frames * rows * cols]
    pixel_array = pixel_array.reshape((n_frames, rows, cols))

    frame_segment_numbers = [
        f.SegmentIdentificationSequence[0].ReferencedSegmentNumber for f in per_frame
    ]

    lung_mask = None

    for seg in segments:
        if label in seg.SegmentLabel:
            seg_num = seg.SegmentNumber
            seg_frames_idx = [
                i for i, s_num in enumerate(frame_segment_numbers) if s_num == seg_num
            ]

            if lung_mask is None:

                lung_mask = pixel_array[seg_frames_idx]

            else:

                lung_mask += pixel_array[seg_frames_idx]

    lung_mask = torch.from_numpy(lung_mask > 0) # type: ignore

    D, H, W = lung_mask.shape

    coords = lung_mask.nonzero(as_tuple=False)
    if coords.numel() == 0:
        raise ValueError("No bounding box to extract.")

    z_min, y_min, x_min = coords.min(dim=0).values
    z_max, y_max, x_max = coords.max(dim=0).values

    def pad_dim(vmin, vmax, pad_val, r_lim):
        lpad = pad_val//2
        rpad = pad_val - lpad
        vmin = max(0, vmin - lpad)
        vmax = min(r_lim, vmax + rpad)

        return vmin, vmax
    
    z_min, z_max = pad_dim(z_min, z_max, pad, D)
    y_min, y_max = pad_dim(y_min, y_max, pad, H)
    x_min, x_max = pad_dim(x_min, x_max, pad, W)
    
    new_H = y_max - y_min
    new_W = x_max - x_min
    H_pad = max(new_H, new_W) - new_H
    W_pad = max(new_H, new_W) - new_W

    y_min, y_max = pad_dim(y_min, y_max, H_pad, H)
    x_min, x_max = pad_dim(x_min, x_max, W_pad, W)

    bbox = (slice(z_min, z_max + 1), slice(y_min, y_max + 1), slice(x_min, x_max + 1))

    return bbox


In [ ]:
patient_id, study_path, rel_seg_path = metadata.row(0)
dcm_path = os.path.join(data_path, study_path)
seg_path = os.path.join(data_path, rel_seg_path)
img, spacing = load_dicom(dcm_path)

tumour_bbox = get_bbox(seg_path, label="Neoplasm", pad=10)

view_volume(img[tumour_bbox], spacing)
print(img[tumour_bbox].shape)

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_79999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 224,
    patch_size = 14,
    device="cuda",
    block_size=64,
    no_crop=True,
)
output_path = "/scratch/VM/radio-foundation/cache/embeddings/NSCLC_Radiomics"
os.makedirs(output_path, exist_ok=True)

for patient_id, rel_dcm_path, rel_seg_path in tqdm(metadata.iter_rows()):
    dcm_path = os.path.join(data_path, rel_dcm_path)
    seg_path = os.path.join(data_path, rel_seg_path)
    
    img, spacing = load_dicom(dcm_path)
    tumour_bbox = get_bbox(seg_path, label="Neoplasm", pad=10)

    img = img[tumour_bbox]
    
    collated_features = generate_embeddings(
        img,
        model=model,
        **data_kwargs # type: ignore
    )

    torch.save(collated_features, os.path.join(output_path, f"{patient_id}.pth"))
